# 생성 쪽 채점 실험 (2026-08-03, 방학 중 개인 연구)

7월 29일 발표 때 생성을 처음 붙여봤는데 점수가 0.018만 움직였다. 그때는 표본이 둘뿐이라
"이겼다"고 말하지 못하고 "못 잡는다"로만 발표했다. 방학에 다시 보니 **못 잰 게 아니라 잴 도구가
없었던 것**이었다. 이 노트북은 그 도구를 만들면서 잰 것들을 순서대로 담았다.

결론부터 적으면 이렇다.

1. 우리는 **검색에는 정답이 있는데 생성에는 채점자가 없었다**
2. 채점자를 만들고 나서 보니 더 앞에 문제가 있었다 — **4컷이 서로를 모른다**
3. 잰 것 다섯 중 셋은 기각됐고, **기각된 데서 발표에 쓸 숫자가 나왔다**

v1 동결본은 하나도 안 건드렸다. 여기 있는 건 전부 새로 만든 것이다.

## 돌리기 전에

임베딩 캐시(`verify/ink_*.npy`)가 있어야 돈다. 용량 때문에 저장소에 안 올렸다 —
만드는 법은 옆에 있는 `실행_안내.md` 를 보면 된다. **생성 API 는 한 번도 안 쓴다** (돈 안 든다).

In [1]:
# 오늘 쓰는 것 전부. 무거운 건 임베딩 두 개뿐이다.
import numpy as np
import itertools, math

TOPK = 5          # 이웃 몇 장을 보고 라벨을 정할 것인가 (score.py 와 같은 값)
K    = 4          # 한 편은 4컷

# 라벨과 원본ID. 원본ID는 "같은 그림을 다른 기법으로 그린 것"을 이웃에서 빼는 데 쓴다
style   = np.load("verify/ink_style.npy")
content = np.load("verify/ink_content.npy", allow_pickle=True)

def load_emb(path):
    """임베딩을 읽고 L2 정규화한다. 정규화해야 내적이 곧 코사인이 된다."""
    E = np.load(path).astype("float32")
    return E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-8)

E_clip = load_emb("verify/ink_emb.npy")
E_gram = load_emb("verify/ink_gram.npy")

n_class = len(np.unique(style))
CHANCE  = 1 / n_class

print(f"코퍼스 {len(style)}장 / 그림체 {n_class}종 / 원본 {len(np.unique(content))}종")
print(f"CLIP {E_clip.shape}  Gram {E_gram.shape}")
print(f"무작위로 찍었을 때 기대값 = 1/{n_class} = {CHANCE:.3f}")

코퍼스 591장 / 그림체 9종 / 원본 200종
CLIP (591, 512)  Gram (591, 174560)
무작위로 찍었을 때 기대값 = 1/9 = 0.111


---

## 1. 검색에는 정답이 있고 생성에는 채점자가 없었다

우리 파이프라인을 검색과 생성으로 갈라 놓고 보면 이렇게 비어 있었다.

| | 검색 | 생성 |
|---|---|---|
| 정답 | `meta.csv` 의 `style` 컬럼 (m1~m9) | "m3로 그려줘" 라는 요청문 |
| 정답과 대조하는 수단 | `p@5` | **없음** |
| 표본 | 1,259장 | n=2 |

정답은 사실 **양쪽 다 있었다.** 없었던 건 **생성물에 라벨을 붙이는 방법**이다.
코퍼스 그림에는 `meta.csv` 가 라벨을 달아주는데, 방금 생성된 그림에는 아무도 안 달아준다.

그래서 **역방향 채점**을 쓰기로 했다. 생성물을 코퍼스에 질의로 던져 가장 가까운 5장을 찾고,
그 5장의 라벨로 생성물을 판정한다. 검색기를 라벨 붙이는 도구로 재활용하는 것이라
새 모델도 새 지표도 필요 없다.

In [2]:
def sim_matrix(E):
    """유사도 행렬. 자기 자신과 **같은 원본**은 후보에서 뺀다.

    안 빼면 그림체가 아니라 "내용이 같은 것"을 찾고도 맞힌 것이 된다.
    score.py 가 쓰는 규칙과 같다.
    """
    S = E @ E.T
    S[content[:, None] == content[None, :]] = -np.inf
    return S

def predict_labels(S):
    """이웃 TOPK 장의 최다 득표 라벨을 그 그림의 판정 결과로 삼는다. = 역방향 채점"""
    pred = np.empty(len(S), dtype=style.dtype)
    for i in range(len(S)):
        top = np.argpartition(-S[i], TOPK)[:TOPK]
        top = top[np.argsort(-S[i][top])][:TOPK]
        lab, cnt = np.unique(style[top], return_counts=True)
        pred[i] = lab[np.argmax(cnt)]
    return pred

S_clip, S_gram = sim_matrix(E_clip), sim_matrix(E_gram)
pred_clip, pred_gram = predict_labels(S_clip), predict_labels(S_gram)

print("역방향 채점이 코퍼스 그림 자신을 맞히는 비율 (컷 단위 라벨 정확도)")
print(f"  무작위  {CHANCE:.3f}")
print(f"  CLIP    {(pred_clip == style).mean():.3f}")
print(f"  Gram    {(pred_gram == style).mean():.3f}")

역방향 채점이 코퍼스 그림 자신을 맞히는 비율 (컷 단위 라벨 정확도)
  무작위  0.111
  CLIP    0.296
  Gram    0.440


---

## 2. 실험 1 — 컷 간 코사인 유사도로 그림체를 가릴 수 있나

"4컷이 서로 같은 그림체인가"를 재는 제일 단순한 방법은 컷끼리 코사인 유사도를 평균내는 것이다.
6쌍(4개 중 2개 고르기)의 평균을 한 편의 점수로 삼았다.

진짜 그림으로 두 가지를 만들어 비교했다. **같은 그림체 4장**(원본은 서로 다르게)과
**다른 그림체 4장**이다. 앞엣것이 우리가 원하는 상태, 뒤엣것이 4컷이 따로 노는 상태다.

In [3]:
RNG = np.random.default_rng(20260803)
TRIALS = 2000

def edition_cosine(E, idx):
    """한 편(4컷)의 일관성 점수 = 6쌍 코사인의 평균"""
    V = E[idx]
    S = V @ V.T
    return float(S[np.triu_indices(len(idx), k=1)].mean())

def sample_same_style():
    """같은 그림체 4장. 단 원본은 서로 다르게 뽑는다.
    같은 원본이 섞이면 '그림체가 같아서'가 아니라 '내용이 같아서' 점수가 오른다."""
    for _ in range(200):
        s = RNG.choice(np.unique(style))
        pool = np.where(style == s)[0]
        if len(pool) < K:
            continue
        seen, idx = set(), []
        for i in RNG.permutation(pool):
            if content[i] in seen:
                continue
            seen.add(content[i]); idx.append(int(i))
            if len(idx) == K:
                return idx, s
    return None, None

def sample_diff_style():
    ss = RNG.choice(np.unique(style), size=K, replace=False)
    return [int(RNG.choice(np.where(style == s)[0])) for s in ss]

same, diff = [], []
for _ in range(TRIALS):
    idx, _ = sample_same_style()
    if idx:
        same.append(edition_cosine(E_clip, idx))
    diff.append(edition_cosine(E_clip, sample_diff_style()))

for name, a in [("같은 그림체 4장", same), ("다른 그림체 4장", diff)]:
    a = np.array(a)
    p = np.percentile(a, [5, 50, 95])
    print(f"{name}   중앙 {p[1]:.3f}   5~95% [{p[0]:.3f} ~ {p[2]:.3f}]   n={len(a)}")

print(f"\n중앙값 차이 {np.median(same) - np.median(diff):+.3f}")
print("같은 그림 4장(복붙)이면 얼마가 나오나:",
      f"{edition_cosine(E_clip, [7]*K):.3f}")

같은 그림체 4장   중앙 0.669   5~95% [0.584 ~ 0.773]   n=2000
다른 그림체 4장   중앙 0.656   5~95% [0.578 ~ 0.736]   n=2000

중앙값 차이 +0.013
같은 그림 4장(복붙)이면 얼마가 나오나: 1.000


### 실패했다

차이가 0.015고 분포가 거의 완전히 겹친다. **이 지표로는 그림체를 못 가린다.**

수묵화끼리는 코사인이 죄다 0.65 근처다. "수묵화다"라는 공통 신호가 자릿수를 다 먹어서
그림체 차이는 소수점 둘째 자리에서 논다. 공교롭게도 0.015는 발표 때 문제였던 0.018과 같은 크기다.

**그래도 버리진 않았다.** 같은 그림 4장이면 1.000이 나오니, **복붙 탐지 상한**으로 쓴다.
정상 범위가 0.584~0.770이므로 `0.95 초과`면 "일관성이 높은 게 아니라 복붙"으로 막는다.

---

## 3. 실험 2 — 절대값 말고 순위로 보면 갈린다

앞에서 실패한 건 **절대값**을 봤기 때문이다. 같은 임베딩이라도 이웃을 찾아 **순위**로 보면
무작위의 2.7~4.0배가 나온다(위 1번 셀 결과). 그래서 편 단위 판정도 순위 기반으로 다시 짰다.

편 하나에 지표 두 개를 낸다.

- `style_precision` — 4컷 중 **요청한 라벨**로 판정된 컷의 비율
- `all_agree` — 4컷이 **전부 같은 라벨**로 판정됐나 (0 또는 1)

In [4]:
def edition_scores(pred, idx, requested):
    labels = pred[idx]
    _, cnt = np.unique(labels, return_counts=True)
    return {"style_precision": float((labels == requested).mean()),
            "all_agree": int(cnt.max() == K)}

rows = {}
for name, pred in [("CLIP", pred_clip), ("Gram", pred_gram)]:
    RNG_ = np.random.default_rng(20260803)
    globals()["RNG"] = RNG_          # 두 채점기가 같은 표본을 보게 시드를 맞춘다
    sp_same, ag_same, ag_diff = [], [], []
    for _ in range(TRIALS):
        idx, s = sample_same_style()
        if idx:
            r = edition_scores(pred, idx, s)
            sp_same.append(r["style_precision"]); ag_same.append(r["all_agree"])
        d = sample_diff_style()
        _, cntd = np.unique(pred[d], return_counts=True)
        ag_diff.append(int(cntd.max() == K))
    rows[name] = (np.array(sp_same), np.array(ag_same), np.array(ag_diff))

print(f"{'':8s} {'컷 라벨정확도':>13s} {'편 style_prec 중앙':>19s} {'4컷 전부일치':>13s} {'다른그림체 전부일치':>19s}")
print(f"{'무작위':8s} {CHANCE:13.3f} {CHANCE:19.3f} {'~0':>13s} {'-':>19s}")
for name, pred in [("CLIP", pred_clip), ("Gram", pred_gram)]:
    sp, ag, agd = rows[name]
    print(f"{name:8s} {(pred == style).mean():13.3f} {np.median(sp):19.3f} "
          f"{ag.mean():13.3f} {agd.mean():19.3f}")

               컷 라벨정확도     편 style_prec 중앙       4컷 전부일치          다른그림체 전부일치
무작위              0.111               0.111            ~0                   -
CLIP             0.296               0.250         0.021               0.004
Gram             0.440               0.500         0.065               0.000


### 채점은 CLIP 말고 Gram으로 해야 한다

Gram 이 모든 항목에서 낫다. 특히 **다른 그림체 4장이 전부 같은 라벨로 판정된 경우가
2000번 중 0번**이다 — 헛통과가 없다는 뜻이다.

CLIP 은 편 style precision 중앙값이 0.250이라 무작위 0.111과의 폭이 0.14뿐이다.
**잴 공간이 없다.** Gram 은 0.500이라 폭이 두 배다.

---

## 4. 진짜 그림도 만점을 못 받는다 — 천장이 0.500이다

위 표에서 중요한 건 Gram 의 0.500이 **진짜 그림 4장을 넣었을 때** 나온 값이라는 점이다.
코퍼스에 실제로 존재하는 진품이 그 점수다. **생성물이 이걸 넘을 수 없다.**

그러니 점수를 절대값으로 읽으면 안 되고 천장 대비로 정규화해야 한다.

```
정규화 = (측정값 - 무작위) / (천장 - 무작위)
1.0 이면 진품만큼, 0 이면 무작위와 같음
```

통과선도 이 분포를 보고 정한다. 무작위가 통과할 확률은 이항분포로 계산했다.

In [5]:
sp_gram = rows["Gram"][0]
q = np.percentile(sp_gram, [10, 25, 50, 75, 90])
print("Gram 기준 진짜 그림 4장짜리 편의 style_precision 분포")
print(f"  10% {q[0]:.3f} / 25% {q[1]:.3f} / 중앙 {q[2]:.3f} / 75% {q[3]:.3f} / 90% {q[4]:.3f}\n")

def chance_pass(thr):
    """무작위로 찍었을 때 style_precision 이 thr 이상 나올 확률. 이항분포 B(4, 1/9)."""
    need = math.ceil(thr * K - 1e-9)
    return sum(math.comb(K, i) * CHANCE**i * (1 - CHANCE)**(K - i) for i in range(need, K + 1))

print(f"{'통과선':>18s} {'진짜 그림 통과':>14s} {'무작위 통과':>12s}")
for thr, note in [(0.25, "4컷 중 1컷"), (0.50, "4컷 중 2컷"), (0.75, "4컷 중 3컷")]:
    print(f"  {thr:.2f} ({note})   {(sp_gram >= thr).mean():12.3f} {chance_pass(thr):12.3f}")

print("\n-> 0.25 는 무작위도 37% 가 통과해서 게이트 노릇을 못 한다. **통과선은 0.50**")

Gram 기준 진짜 그림 4장짜리 편의 style_precision 분포
  10% 0.000 / 25% 0.250 / 중앙 0.500 / 75% 0.500 / 90% 0.750

               통과선       진짜 그림 통과       무작위 통과
  0.25 (4컷 중 1컷)          0.844        0.376
  0.50 (4컷 중 2컷)          0.538        0.064
  0.75 (4컷 중 3컷)          0.249        0.005

-> 0.25 는 무작위도 37% 가 통과해서 게이트 노릇을 못 한다. **통과선은 0.50**


---

## 5. 실험 3 — 채점기를 여럿 모아 투표시키면 나아지나

앙상블이 먹히려면 **서로 다른 데서 틀려야** 한다. 다 같은 데서 틀리면 몇 개를 모아도 결과가 같다.
먼저 그 조건부터 확인했다.

In [6]:
a, b = (pred_clip == style), (pred_gram == style)
print("맞춤/틀림 교차표")
print(f"  둘 다 맞힘     {(a & b).mean():.3f}")
print(f"  CLIP 만 맞힘   {(a & ~b).mean():.3f}   <- Gram 이 놓친 걸 CLIP 이 건짐")
print(f"  Gram 만 맞힘   {(~a & b).mean():.3f}")
print(f"  둘 다 틀림     {(~a & ~b).mean():.3f}  <- 투표로 못 구한다")

upper = (a | b).mean()
best  = max(a.mean(), b.mean())
print(f"\n한쪽이라도 맞힌 비율(이론상 상한) {upper:.3f}")
print(f"단독 최고                      {best:.3f}")
print(f"-> 투표로 건질 여지             {upper - best:+.3f}")

맞춤/틀림 교차표
  둘 다 맞힘     0.173
  CLIP 만 맞힘   0.124   <- Gram 이 놓친 걸 CLIP 이 건짐
  Gram 만 맞힘   0.267
  둘 다 틀림     0.437  <- 투표로 못 구한다

한쪽이라도 맞힌 비율(이론상 상한) 0.563
단독 최고                      0.440
-> 투표로 건질 여지             +0.124


In [7]:
def zscore(S):
    """채점기마다 유사도 스케일이 다르다. 더하려면 같은 자로 맞춰야 한다."""
    ok = np.isfinite(S)
    Z = np.full_like(S, -np.inf)
    Z[ok] = (S[ok] - S[ok].mean()) / (S[ok].std() + 1e-8)
    return Z

Zc, Zg = zscore(S_clip), zscore(S_gram)
print("soft voting — 유사도를 z 정규화해 가중합한 뒤 이웃을 찾는다")
print(f"  {'가중치(CLIP:Gram)':>20s} {'정확도':>8s}")
print(f"  {'Gram 단독':>20s} {b.mean():8.3f}")
best_ens = None
for w in [0.5, 1.0, 2.0, 4.0]:
    acc = (predict_labels(Zc + w * Zg) == style).mean()
    print(f"  {'1 : ' + str(w):>20s} {acc:8.3f}")
    if w == 4.0:
        best_ens = (predict_labels(Zc + w * Zg) == style).astype(float)

soft voting — 유사도를 z 정규화해 가중합한 뒤 이웃을 찾는다
        가중치(CLIP:Gram)      정확도
               Gram 단독    0.440
               1 : 0.5    0.374
               1 : 1.0    0.384
               1 : 2.0    0.416
               1 : 4.0    0.455


In [8]:
# 1:4 가 단독보다 조금 높다. 그런데 그 차이가 진짜인가 — 짝지어 비교로 판정한다
d = best_ens - b.astype(float)
rng = np.random.default_rng(0)
boots = d[rng.integers(0, len(d), (10000, len(d)))].mean(axis=1)
lo, hi = np.percentile(boots, [2.5, 97.5])

print(f"앙상블(1:4) - Gram 단독")
print(f"  차이            {d.mean():+.4f}")
print(f"  차이의 95% 구간 [{lo:+.4f} ~ {hi:+.4f}]")
print(f"  질의별 승패     앙상블승 {int((d>0).sum())} / Gram승 {int((d<0).sum())} / 무 {int((d==0).sum())}")
print("\n판정:", "앙상블 승" if lo > 0 else ("Gram 승" if hi < 0 else "아직 모른다 — 구간이 0을 품는다"))

앙상블(1:4) - Gram 단독
  차이            +0.0152
  차이의 95% 구간 [-0.0254 ~ +0.0558]
  질의별 승패     앙상블승 80 / Gram승 71 / 무 440

판정: 아직 모른다 — 구간이 0을 품는다


### 기각. 그런데 남은 숫자가 더 값지다

조건(오류 비겹침)은 만족했는데 실제로는 나빠졌다. 원인은 **실력 차**다.
CLIP 0.296과 Gram 0.440은 차이가 커서 **약한 표가 강한 표를 희석시킨다.**
오류가 안 겹치는 건 필요조건이지 충분조건이 아니었다.

남은 수확: **`둘 다 틀림 0.437`.** 임베딩 두 개를 합쳐도 44%는 아무도 못 맞힌다.
채점기를 더 얹어서 풀 문제가 아니라 **문제 자체가 어렵다는 증거**다.
이건 천장 0.500과 서로 다른 각도에서 같은 말을 하고 있다.

---

## 6. 실험 4 — 판정 말고 선택에 쓰면 확 오른다

같은 Gram 채점기인데 질문을 바꿔봤다.
"이게 m3인가?"(절대 판정) 대신 **"후보 N장 중 뭐가 제일 m3다운가?"**(상대 비교)로.

"m3로 그려줘" 하고 N장을 받았는데 그중 진짜 m3는 한 장뿐인 상황을 코퍼스로 흉내낸다.

In [9]:
targets = np.unique(style)
t_index = {t: j for j, t in enumerate(targets)}

# 그림마다 "각 라벨다움" 점수를 미리 계산 (이웃 TOPK 중 그 라벨의 비율)
CS = np.zeros((len(S_gram), len(targets)), dtype="float32")
for i in range(len(S_gram)):
    top = np.argpartition(-S_gram[i], TOPK)[:TOPK]
    top = top[np.argsort(-S_gram[i][top])][:TOPK]
    lab, cnt = np.unique(style[top], return_counts=True)
    for l, c in zip(lab, cnt):
        CS[i, t_index[l]] = c / TOPK

rng = np.random.default_rng(20260803)
print(f"{'N':>3s} {'진짜를 고를 확률':>16s} {'무작위':>8s} {'배수':>7s}")
for N in (2, 3, 5):
    win = 0
    for _ in range(3000):
        target = rng.choice(targets); j = t_index[target]
        real = int(rng.choice(np.where(style == target)[0]))
        others = [int(rng.choice(np.where(style == s)[0]))
                  for s in rng.choice(targets[targets != target], size=N-1, replace=False)]
        cand = [real] + others
        sc = CS[cand, j]
        best = int(rng.choice(np.flatnonzero(sc == sc.max())))   # 동점이면 무작위
        win += (cand[best] == real)
    print(f"{N:3d} {win/3000:16.3f} {1/N:8.3f} {(win/3000)/(1/N):7.2f}")

  N        진짜를 고를 확률      무작위      배수


  2            0.795    0.500    1.59


  3            0.704    0.333    2.11


  5            0.581    0.200    2.90


### 판정 0.440이 선택 0.693이 됐다

모델도 임베딩도 안 바꾸고 **질문만 바꿔서** 얻은 차이다.
절대 판정은 임계값이 필요한데 그 선이 클래스마다 다르고, 상대 비교는 선이 필요 없어서다.

여기서 실무 규칙이 하나 나온다.

> **채점기가 약할 때는 버리는 데 쓰면 안 되고 고르는 데 써야 한다.**

정확도 0.440은 뒤집으면 **진짜 그림의 56%를 "아니다"라고 잘못 짚는다**는 뜻이다.
"실패하면 재생성" 루프를 돌리면 잘 그린 걸 버리게 되어 돌수록 나빠질 수 있다.
반면 고르기는 채점기가 엉터리여도 최악이 무작위 뽑기라 밑질 게 없다.

빌드업 노트북(`03_빌드업/빌드업_랭그래프_오토젠.ipynb`)에 넣어둔 재생성 루프는
**지금 채점기로는 쓰면 안 된다.** 대신 컷당 3장 뽑아 고르는 방식을 제안한다.
생성비가 3배(편당 0.18달러에서 0.55달러)로 늘지만, 오늘 잰 것 중 노이즈 바닥을
확실히 넘은 유일한 개선이다.

N을 키우면 배수는 오르는데 절대 성공률은 떨어진다(후보가 많을수록 헷갈린다).
비용도 N배라 **N=3이 균형점**이다.

---

## 7. 품질 게이트 — 지금까지 잰 것을 판정 규칙으로 묶기

편 하나가 내보낼 만한가를 판정한다. 통과/탈락과 함께 **어디를 고쳐야 하는지**까지 낸다.

흔한 품질 게이트와 두 군데를 다르게 만들었다.

1. **임계값이 양방향이다.** 보통은 "최소 몇 이상"만 보는데, 컷 간 유사도에는 **상한**이 필요하다.
   4컷이 완전히 똑같으면 1.000으로 만점인데 그건 복붙이라 실패다
2. **천장 대비로 정규화한다.** "정확도 0.95 이상" 같은 절대선은 천장이 1.0일 때 쓰는 것이다.
   우리는 진짜 그림도 0.500이라 절대선이 무의미하다

In [10]:
import sys, json
for p in (".", "kit"):        # 저장소에서는 옆에, 작업 폴더에서는 kit/ 에 있다
    if p not in sys.path:
        sys.path.insert(0, p)
from gate import check_quality_gate, summarize, EXAMPLE_REPORT

# 손으로 만든 예시 6편으로 게이트가 제대로 막고 통과시키는지 본다
cases = [
    dict(EXAMPLE_REPORT),                                                # 정상
    {**EXAMPLE_REPORT, "case_id": "G-m3-02", "panels_produced": 3},      # 컷 누락
    {**EXAMPLE_REPORT, "case_id": "G-m5-01", "style_precision": 0.25},   # 그림체 실패
    {**EXAMPLE_REPORT, "case_id": "G-m2-01", "edition_cosine": 0.985},   # 복붙
    {**EXAMPLE_REPORT, "case_id": "G-m4-01", "style_precision": 0.75},   # 진품 초과
    {**EXAMPLE_REPORT, "case_id": "G-m7-01", "latency_sec": 180},        # 너무 느림
]
results = [check_quality_gate(c) for c in cases]
for r in results:
    print(f"[{'통과' if r['can_deploy'] else '탈락'}] {r['case_id']}  정규화 {r['style_normalized']:+.2f}")
    for f in r["failed_reasons"]:
        print(f"        X {f}")
    for w in r["warnings"]:
        print(f"        ! {w}")

print()
print(json.dumps(summarize(results), ensure_ascii=False, indent=1))

[통과] G-m3-01  정규화 +1.00
[탈락] G-m3-02  정규화 +1.00
        X 컷이 빠졌다 — 3/4컷 (recall 0.75). 고칠 곳: 출력 스키마 검증
[탈락] G-m5-01  정규화 +0.36
        X 그림체가 안 잡혔다 — style precision 0.25 < 0.50 (무작위 0.111 / 천장 0.50). 고칠 곳: 레퍼런스 선택
[탈락] G-m2-01  정규화 +1.00
        X 4컷이 사실상 같은 그림이다 — 컷 간 코사인 0.985 > 0.95. 일관성이 높은 게 아니라 복붙이다
[통과] G-m4-01  정규화 +1.64
[통과] G-m7-01  정규화 +1.00
        ! 느리다 — 180초 > 120초. 팔기 어렵다

{
 "편수": 6,
 "strict_accuracy": 0.5,
 "통과": 3,
 "탈락": 3,
 "평균_style_normalized": 1.0,
 "탈락사유": {
  "컷이 빠졌다": 1,
  "그림체가 안 잡혔다": 1,
  "4컷이 사실상 같은 그림이다": 1
 }
}


탈락 사유가 **어디를 고칠지**까지 말해준다. 컷이 빠지면 출력 스키마, 그림체가 안 잡히면
레퍼런스 선택이다. 그리고 `strict_accuracy`(통째로 통과한 편의 비율)를 따로 내는 이유는
**부분점수 평균이 좋아도 쓸 수 있는 편이 없을 수 있어서**다.

---

## 8. 오늘 가장 큰 발견은 채점이 아니라 생성 쪽에 있었다

채점 방법을 만들다가 생성 코드를 들여다봤는데, **컷마다 독립으로 API를 부르고 있었다.**
매 컷이 같은 레퍼런스 3장만 보고 그려진다. 앞 컷이 다음 컷에 안 들어간다.

```
지금        컷1 <- [레퍼런스 3장]
            컷2 <- [레퍼런스 3장]      서로 모름
            컷3 <- [레퍼런스 3장]
            컷4 <- [레퍼런스 3장]

바꾸면      컷1 <- [레퍼런스 3장]
            컷2 <- [레퍼런스 3장 + 컷1]
            컷3 <- [레퍼런스 3장 + 컷1]
            컷4 <- [레퍼런스 3장 + 컷1]
```

컷 간 일관성이 안 나오는 게 당연했다. **채점으로 잡을 문제가 아니라 생성 구조 문제**였다.
하루 종일 "어떻게 잴까"를 팠는데 그 앞에 "왜 안 되는가"가 있었다.

비용은 이미지 입력이 3장에서 4장으로 느는 것뿐이라 컷당 0.002달러 정도다.
실측 장당 0.046달러와 비교하면 사실상 공짜다. 그리고 여기가 **랭그래프의 state 가
실제로 쓰이는 자리**이기도 하다 — 앞 노드 결과를 state 에 담아 다음 노드로 넘기면 된다.

확인할 실험 코드를 `ab_prev_cut.py` 로 짜뒀다. 검색기 교체와 앞 컷 물리기를 2x2 로 돌린다.
둘을 한꺼번에 바꾸면 뭐가 효과였는지 못 가리기 때문이다. **아직 안 돌렸다** (돈이 든다).

In [11]:
# 계획과 예상 비용만 확인한다. API 는 안 부른다
import subprocess, sys, os
if os.path.exists("build/ab_prev_cut.py"):
    print(subprocess.run([sys.executable, "build/ab_prev_cut.py", "--dry-run"],
                         capture_output=True, text=True).stdout)
else:
    print("ab_prev_cut.py 는 이 노트북 옆에 있다. 실행하려면 작업 폴더에서 --dry-run 부터.")

조건 4개 x 그림체 2종 x 반복 2회 = 16편
이미지 64장 / 예상 비용 약 $2.94 / 예상 시간 약 20분
조건: clip+독립, clip+순차, gram+독립, gram+순차

--dry-run 이라 여기서 멈춘다. 실제로 돌리려면 --dry-run 을 빼라.



---

## 9. 한계 — 읽는 사람이 알아야 할 것

- **위 숫자는 591장 임시 코퍼스 기준이다.** 본 코퍼스(1,259장)로 재면 달라질 수 있다.
  천장 0.500과 통과선 0.50은 설계 검증용이지 확정값이 아니다
- **생성물로는 한 번도 안 재봤다.** 전부 진짜 그림끼리 흉내낸 것이다. 생성물이 코퍼스와 다른
  성질(종이 질감이 없다든지)을 가지면 채점기가 다르게 반응할 수 있다
- **"검색 부품이 생성 부품의 발목을 잡는다"는 아직 가설이다.** 확인하려면 생성물 골든셋이
  필요하고 그건 `ab_prev_cut.py` 를 돌려야 나온다. 발표에서는 "생성 부품은 아직 못 쟀습니다"
  까지만 말해야 한다
- 앞 컷 물리기는 **오류가 전파되는 대가**가 있다. 지금은 컷이 독립이라 하나 망해도 나머지는
  사는데, 순차로 묶으면 컷1이 망하면 4컷이 다 망한다. 그래서 컷1에는 best-of-N 을 같이 걸어야 한다

## 다음에 할 것

1. `ab_prev_cut.py` 돌리기 — 16편, 약 3달러, 20분
2. 게이트 임계값을 본 코퍼스 기준으로 다시 재기
3. 이유를 말하는 채점기가 가능한지 확인 (그림을 보고 "먹 번짐이 부족하다"처럼 말해주는 것).
   되면 그때 되돌아가는 루프를 붙일 수 있다